# LangGraph Research Notes with Persistent State

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that accumulates research notes over multiple steps — gathering sources, extracting claims, cross-referencing, synthesizing — and persists the full state between runs using a checkpoint store so that research can be resumed later.

**Why this matters:** Real research is iterative. You read a paper, take notes, find a contradiction in another paper, revise your notes, discover a gap, and search again. A single-shot pipeline loses everything. A **persistent state graph** remembers where you stopped, what you found, and what's still open — across sessions.

**Pipeline:**

```
  START
    │
    │  STATE: {topic, notes=[], sources=[], ...}
    ▼
  ┌───────────────────────────┐
  │  gather_sources            │  find relevant sources on the topic
  └──────────┬────────────────┘
             │  STATE += {sources: [...]}
             ▼
  ┌───────────────────────────┐
  │  extract_claims            │  pull key claims from each source
  └──────────┬────────────────┘
             │  STATE += {claims: [...]}
             ▼
  ┌───────────────────────────┐
  │  cross_reference           │  check for agreements/contradictions
  └──────────┬────────────────┘
             │  STATE += {cross_ref_report}
             ▼
  ┌───────────────────────────┐
  │  synthesize_notes          │  write structured research notes
  └──────────┬────────────────┘
             │  STATE += {synthesis}
             ▼
  ┌───────────────────────────┐
  │  identify_gaps             │  find open questions / missing evidence
  └──────────┬────────────────┘
        conditional
      ┌─────┴──────┐
      ▼            ▼
   no gaps       gaps found
      │            │
      │            └→ gather_sources (loop with new sub-queries)
      ▼
  ┌───────────────────────────┐
  │  compile_report            │  final structured research report
  └──────────┬────────────────┘
             ▼
           END
```

**Stack:** `LangGraph` (with `MemorySaver` checkpointer), `ChatOllama` + `qwen3.5:9b`, `pandas`

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **graph state vs. memory vs. persistence** — three distinct concepts |
| 2 | Use **list-accumulation fields** in TypedDict state (append, never overwrite) |
| 3 | Implement **MemorySaver** for cross-session checkpoint persistence |
| 4 | Build a **research loop** that feeds gaps back into the source-gathering node |
| 5 | Resume a graph execution from a saved checkpoint |
| 6 | Inspect checkpoint history to see how state evolved over time |
| 7 | Compare single-pass vs. iterative research on the same topic |

## 3. Memory & Persistence — Core Concepts

```
  ╔══════════════════════════════════════════════════════════════════════╗
  ║  THREE LEVELS OF "MEMORY" IN LANGGRAPH                            ║
  ╠══════════════════════════════════════════════════════════════════════╣
  ║                                                                    ║
  ║  1. GRAPH STATE (within a single run)                              ║
  ║     ─────────────────────────────────                              ║
  ║     The TypedDict that flows through nodes during one invocation.  ║
  ║     Each node reads from and writes to this state.                 ║
  ║     LIFETIME: one graph.invoke() call                              ║
  ║     LOST WHEN: the run completes (unless checkpointed)            ║
  ║                                                                    ║
  ║  2. CHECKPOINTS (persistence across runs)                          ║
  ║     ──────────────────────────────────────                         ║
  ║     Snapshots of the full state saved after each node execution.   ║
  ║     Stored in a CheckpointSaver (e.g., MemorySaver, SqliteSaver). ║
  ║     LIFETIME: until the saver is cleared or the process ends      ║
  ║     (MemorySaver is in-memory; SqliteSaver writes to disk)        ║
  ║     USE CASE: resume a run, replay from a specific step,          ║
  ║     inspect state evolution                                        ║
  ║                                                                    ║
  ║  3. ACCUMULATED STATE (within state via list fields)               ║
  ║     ─────────────────────────────────────────────                  ║
  ║     Lists in the state that GROW across loop iterations.           ║
  ║     E.g., notes=[], sources=[], claims=[] — each iteration adds.  ║
  ║     Combined with checkpoints, this gives you full research        ║
  ║     memory across steps AND sessions.                              ║
  ║                                                                    ║
  ╠══════════════════════════════════════════════════════════════════════╣
  ║  HOW THEY WORK TOGETHER                                            ║
  ║                                                                    ║
  ║  Run 1:                                                            ║
  ║    gather → extract → cross_ref → synthesize → gaps → gather ...  ║
  ║    └─ checkpoint saved after EACH node ─┘                         ║
  ║    └─ sources/claims/notes ACCUMULATE within state ─┘             ║
  ║                                                                    ║
  ║  [process stops — state would normally be lost]                    ║
  ║                                                                    ║
  ║  Run 2 (resume):                                                   ║
  ║    Load checkpoint → state has ALL accumulated data                ║
  ║    Continue from where Run 1 stopped                               ║
  ║                                                                    ║
  ╚══════════════════════════════════════════════════════════════════════╝
```

### Key Distinction: Overwrite vs. Accumulate

```
  OVERWRITE PATTERN (previous notebooks):              ACCUMULATE PATTERN (this notebook):
   cost_analysis = "..."   ← replaced each time        notes = [note1, note2, ...]   ← grows
   exec_summary = "..."    ← one node owns it          sources = [src1, src2, ...]   ← grows
                                                        claims = [c1, c2, c3, ...]    ← grows

  WHEN TO USE:
   Overwrite: field has one definitive value            Accumulate: field collects from loops
   (e.g., final_report)                                 (e.g., research notes from many sources)
```

## 4. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph pandas numpy

print("Dependencies: langchain, langgraph, pandas")
print("LLM: Ollama with qwen3.5:9b (local, no API keys)")

## 5. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict, Annotated
from datetime import datetime

import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Checkpointer: MemorySaver (in-memory persistence)")

## 6. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.3) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — State Schema with Accumulation

## 7. The Reducer Pattern for List Fields

LangGraph uses **reducers** to decide how partial state updates merge into the full state. By default, a field is **overwritten**. To make a list field **accumulate** (append instead of replace), we use the `Annotated` type with an `operator.add` reducer.

```
  WITHOUT reducer (default):                WITH reducer (accumulate):
  ───────────────────────────              ──────────────────────────────
  State before: {notes: ["A", "B"]}        State before: {notes: ["A", "B"]}
  Node returns: {notes: ["C"]}             Node returns: {notes: ["C"]}
  State after:  {notes: ["C"]}             State after:  {notes: ["A", "B", "C"]}
                     ↑ overwritten!                            ↑ appended!

  HOW: Use Annotated[list, operator.add] in the TypedDict.
  This tells LangGraph: "when merging, concatenate lists."
```

In [ ]:
import operator


class ResearchState(TypedDict):
    """State schema for the research notes pipeline.

    List fields use Annotated[list, operator.add] so partial updates
    APPEND to existing lists instead of overwriting them.
    """
    # ── Immutable inputs ──
    topic: str                                        # research topic
    sub_queries: Annotated[list, operator.add]        # search queries (grows)

    # ── Accumulated fields (APPEND, never overwrite) ──
    sources: Annotated[list, operator.add]            # discovered sources
    claims: Annotated[list, operator.add]             # extracted claims
    notes: Annotated[list, operator.add]              # research notes

    # ── Overwrite fields (latest value wins) ──
    cross_ref_report: str                             # latest cross-reference
    synthesis: str                                    # latest synthesis
    gaps: str                                         # identified gaps
    gap_queries: list                                 # new queries from gaps
    has_gaps: bool                                    # routing flag
    iteration: int                                    # loop counter
    final_report: str                                 # compiled report
    current_node: str                                 # debugging


print("State schema: ResearchState")
print()
print("Accumulating fields (operator.add reducer):")
print("  sources      — each gather_sources call ADDS new sources")
print("  claims       — each extract_claims call ADDS new claims")
print("  notes        — each synthesize call ADDS new notes")
print("  sub_queries  — each loop iteration ADDS new queries")
print()
print("Overwrite fields (default):")
print("  synthesis, cross_ref_report, gaps  — latest value wins")

## 8. Research Topics

Two simulated research topics with different depth requirements.

In [ ]:
RESEARCH_TOPICS = [
    {
        "topic": "The impact of retrieval-augmented generation (RAG) on reducing LLM hallucinations",
        "initial_queries": [
            "RAG systems for reducing factual hallucination in LLMs",
            "Comparison of RAG vs fine-tuning for factual accuracy",
            "Limitations and failure modes of RAG pipelines",
        ],
    },
    {
        "topic": "Transfer learning efficiency: when does fine-tuning outperform training from scratch?",
        "initial_queries": [
            "Transfer learning vs training from scratch benchmark studies",
            "Domain shift and negative transfer in pre-trained models",
            "Few-shot vs full fine-tuning trade-offs",
        ],
    },
]

print(f"Research topics: {len(RESEARCH_TOPICS)}")
for rt in RESEARCH_TOPICS:
    print(f"  Topic: {rt['topic'][:60]}...")
    print(f"    Initial queries: {len(rt['initial_queries'])}")

---

# Part B — Build the Graph Nodes

## 9. Node 1: Gather Sources

Simulates finding research sources for each query. In production this would call a search API — here the LLM generates plausible source references.

```
  Transition: START → gather_sources (or gap loop → gather_sources)
  Precondition:  sub_queries has at least one query
  Postcondition: sources list GREW by N new items
  Accumulation:  operator.add → new sources appended to existing
```

In [ ]:
GATHER_SYSTEM = """You are a research librarian. Given a research query, identify
3 relevant sources. Return valid JSON only.

For each source provide:
- title: paper/article title
- authors: author names
- year: publication year
- type: paper | survey | blog | report
- relevance: one sentence on why this is relevant

Return: [{"title":"...","authors":"...","year":2023,"type":"paper","relevance":"..."}]

Generate plausible but CLEARLY SYNTHETIC sources — do NOT claim they are real."""


def gather_sources(state: ResearchState) -> dict:
    """Node: Find sources for each sub-query.

    Transition: START → gather (first run) or gaps → gather (loop)
    Precondition: sub_queries non-empty
    Postcondition: sources list grows (operator.add)
    """
    print("  [NODE] gather_sources")
    iteration = state.get("iteration", 0)

    # On first run, use initial queries. On loop, use gap_queries.
    if iteration == 0:
        queries = state.get("sub_queries", [])[-3:]  # latest 3
    else:
        queries = state.get("gap_queries", [])[:2]  # gap-driven, max 2

    new_sources = []
    for q in queries:
        raw = ask(
            f"Research query: {q}\n\nFind 3 relevant sources.",
            system=GATHER_SYSTEM,
            temperature=0.3,
        )
        parsed = parse_json(raw)
        if parsed and isinstance(parsed, list):
            for src in parsed:
                src["query"] = q
                src["iteration"] = iteration
            new_sources.extend(parsed)
            print(f"    Query '{q[:40]}...' → {len(parsed)} sources")
        else:
            new_sources.append({"title": raw[:100], "query": q, "iteration": iteration})
            print(f"    Query '{q[:40]}...' → 1 source (fallback)")

    prior = len(state.get("sources", []))
    print(f"    Total: {prior} existing + {len(new_sources)} new")

    # Return list → operator.add appends to state["sources"]
    return {"sources": new_sources, "current_node": "gather_sources"}


print("Node defined: gather_sources")
print("  READS:  sub_queries (or gap_queries on loop), iteration")
print("  WRITES: sources (APPEND via operator.add)")

## 10. Node 2: Extract Claims

Pulls key claims from each source. Claims accumulate across iterations so the research grows richer over time.

```
  Transition: gather_sources → extract_claims
  Precondition:  sources list has items
  Postcondition: claims list GREW
  Accumulation:  operator.add → new claims appended
```

In [ ]:
EXTRACT_SYSTEM = """You are a research analyst. Given source descriptions, extract
the key claims or findings. Return valid JSON.

For each claim:
- claim: the assertion (one sentence)
- source: which source it comes from
- confidence: high | medium | low (based on source quality)
- category: finding | methodology | limitation | comparison

Return: [{"claim":"...","source":"...","confidence":"...","category":"..."}]

Extract 2-3 claims per source. Be precise and specific."""


def extract_claims(state: ResearchState) -> dict:
    """Node: Extract key claims from gathered sources.

    Transition: gather_sources → extract_claims
    Precondition: sources has items
    Postcondition: claims list grows (operator.add)
    """
    print("  [NODE] extract_claims")
    iteration = state.get("iteration", 0)

    # Only process sources from the current iteration
    current_sources = [s for s in state.get("sources", []) if s.get("iteration") == iteration]
    if not current_sources:
        current_sources = state.get("sources", [])[-6:]  # fallback: last 6

    sources_text = "\n".join(
        f"- {s.get('title', 'Untitled')} ({s.get('year', '?')}): {s.get('relevance', '')}" 
        for s in current_sources
    )

    raw = ask(
        f"RESEARCH TOPIC: {state['topic']}\n\n"
        f"SOURCES:\n{sources_text}\n\n"
        f"Extract key claims from these sources.",
        system=EXTRACT_SYSTEM,
        temperature=0.2,
    )

    new_claims = parse_json(raw)
    if not new_claims or not isinstance(new_claims, list):
        new_claims = [{"claim": raw[:200], "source": "parse_fallback",
                       "confidence": "low", "category": "finding"}]

    for c in new_claims:
        c["iteration"] = iteration

    prior = len(state.get("claims", []))
    print(f"    Extracted {len(new_claims)} claims from {len(current_sources)} sources")
    print(f"    Total claims: {prior} existing + {len(new_claims)} new")

    return {"claims": new_claims, "current_node": "extract_claims"}


print("Node defined: extract_claims")
print("  READS:  sources, topic, iteration")
print("  WRITES: claims (APPEND via operator.add)")

## 11. Node 3: Cross-Reference Claims

Checks ALL accumulated claims for agreements, contradictions, and patterns — not just the latest batch.

```
  Transition: extract_claims → cross_reference
  Precondition:  claims list has items (all accumulated)
  Postcondition: cross_ref_report (overwrite — latest analysis)
```

In [ ]:
CROSSREF_SYSTEM = """You are a research analyst cross-referencing claims from multiple sources.

Analyze the claims for:
1. AGREEMENTS — claims that reinforce each other
2. CONTRADICTIONS — claims that conflict
3. UNIQUE ANGLES — claims only one source mentions
4. CONFIDENCE PATTERN — do high-confidence claims agree?

Be specific. Cite which claims agree or disagree. This helps the researcher
know what's established vs. debated."""


def cross_reference(state: ResearchState) -> dict:
    """Node: Cross-reference ALL accumulated claims.

    Transition: extract_claims → cross_reference
    Precondition: claims list has items
    Postcondition: cross_ref_report updated (overwrite)
    """
    print("  [NODE] cross_reference")
    all_claims = state.get("claims", [])

    claims_text = "\n".join(
        f"[{c.get('confidence', '?').upper()}] ({c.get('source', '?')}): {c.get('claim', '')}" 
        for c in all_claims[:20]  # cap context window
    )

    report = ask(
        f"RESEARCH TOPIC: {state['topic']}\n\n"
        f"ALL ACCUMULATED CLAIMS ({len(all_claims)} total):\n{claims_text}\n\n"
        f"Cross-reference these claims.",
        system=CROSSREF_SYSTEM,
        temperature=0.2,
    )

    print(f"    Cross-referenced {len(all_claims)} claims")
    print(f"    Report: {len(report)} chars")
    return {"cross_ref_report": report, "current_node": "cross_reference"}


print("Node defined: cross_reference")
print("  READS:  claims (ALL accumulated), topic")
print("  WRITES: cross_ref_report (OVERWRITE — latest analysis)")

## 12. Node 4: Synthesize Notes

Writes structured research notes incorporating cross-reference findings. Notes accumulate — each iteration adds a timestamped entry.

```
  Transition: cross_reference → synthesize_notes
  Precondition:  claims, cross_ref_report exist
  Postcondition: notes list GREW by one entry, synthesis updated
  Accumulation:  notes uses operator.add
```

In [ ]:
SYNTH_SYSTEM = """You are a research note-taker synthesizing findings.
Write a concise research note (150-250 words) that:

1. States what was found in this iteration's sources
2. Notes agreements/contradictions from cross-referencing
3. Highlights the strongest and weakest claims
4. Identifies what's still unclear

Write in note-taking style — bullet points and short paragraphs.
This is a working note, not a final paper."""


def synthesize_notes(state: ResearchState) -> dict:
    """Node: Write structured research notes from current findings.

    Transition: cross_reference → synthesize_notes
    Precondition: claims, cross_ref_report exist
    Postcondition: notes grows, synthesis updated
    """
    print("  [NODE] synthesize_notes")
    iteration = state.get("iteration", 0)

    note_text = ask(
        f"TOPIC: {state['topic']}\n\n"
        f"ITERATION: {iteration}\n"
        f"TOTAL SOURCES: {len(state.get('sources', []))}\n"
        f"TOTAL CLAIMS: {len(state.get('claims', []))}\n\n"
        f"CROSS-REFERENCE REPORT:\n{state['cross_ref_report'][:1200]}\n\n"
        f"Write a research note for this iteration.",
        system=SYNTH_SYSTEM,
        temperature=0.3,
    )

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    note_entry = {
        "iteration": iteration,
        "timestamp": timestamp,
        "note": note_text,
        "sources_count": len(state.get("sources", [])),
        "claims_count": len(state.get("claims", [])),
    }

    prior = len(state.get("notes", []))
    print(f"    Note #{prior + 1}: {len(note_text)} chars")

    # operator.add: [note_entry] will be appended to existing notes
    return {
        "notes": [note_entry],
        "synthesis": note_text,
        "current_node": "synthesize_notes",
    }


print("Node defined: synthesize_notes")
print("  READS:  topic, sources, claims, cross_ref_report, iteration")
print("  WRITES: notes (APPEND), synthesis (OVERWRITE)")

## 13. Node 5: Identify Gaps

Analyzes whether the research is complete or needs another round. Routes the graph.

```
  Transition: synthesize_notes → identify_gaps
  Precondition:  synthesis and notes exist
  Postcondition: has_gaps bool, gap_queries list, iteration incremented

  Routing:
    has_gaps = True  AND iteration < MAX → gather_sources (loop)
    has_gaps = False OR iteration >= MAX  → compile_report
```

In [ ]:
GAPS_SYSTEM = """You are a research advisor reviewing accumulated notes.

Identify GAPS in the current research:
1. Are there assertions without supporting evidence?
2. Are there contradictions that need resolution?
3. Are there obvious sub-topics that haven't been explored?
4. Is the evidence one-sided (only supporting, no counter-arguments)?

Return valid JSON:
{
  "has_gaps": true/false,
  "gap_summary": "overview of what's missing",
  "gap_queries": ["specific search query to fill gap 1", "query 2"],
  "confidence_in_current_notes": "high | medium | low"
}

Be honest. Research is rarely complete after one pass."""


def identify_gaps(state: ResearchState) -> dict:
    """Node: Determine if research needs another iteration.

    Transition: synthesize_notes → identify_gaps
    Precondition: synthesis, notes, claims exist
    Postcondition: has_gaps, gap_queries, iteration updated
    """
    print("  [NODE] identify_gaps")
    iteration = state.get("iteration", 0)

    all_notes = state.get("notes", [])
    notes_text = "\n---\n".join(
        f"Note #{n['iteration']}: {n['note'][:300]}" for n in all_notes[-3:]
    )

    raw = ask(
        f"TOPIC: {state['topic']}\n\n"
        f"ITERATION: {iteration} (sources: {len(state.get('sources', []))}, "
        f"claims: {len(state.get('claims', []))})\n\n"
        f"LATEST NOTES:\n{notes_text}\n\n"
        f"LATEST SYNTHESIS:\n{state.get('synthesis', '')[:800]}\n\n"
        f"Identify remaining gaps.",
        system=GAPS_SYSTEM,
        temperature=0.2,
    )

    result = parse_json(raw)
    if result:
        has_gaps = result.get("has_gaps", False)
        gap_queries = result.get("gap_queries", [])
        gaps_text = result.get("gap_summary", "")
        confidence = result.get("confidence_in_current_notes", "?")
    else:
        has_gaps = False
        gap_queries = []
        gaps_text = raw[:300]
        confidence = "unknown"

    print(f"    Gaps found: {has_gaps} | Confidence: {confidence}")
    if gap_queries:
        print(f"    New queries: {gap_queries[:2]}")

    return {
        "gaps": gaps_text,
        "gap_queries": gap_queries[:2],  # cap new queries
        "has_gaps": has_gaps,
        "iteration": iteration + 1,
        "current_node": "identify_gaps",
    }


print("Node defined: identify_gaps")
print("  READS:  topic, notes, synthesis, sources, claims")
print("  WRITES: gaps, gap_queries, has_gaps, iteration")

## 14. Node 6: Compile Report

Terminal node — assembles all accumulated research into a final report.

```
  Transition: identify_gaps → compile_report (when complete or at cap)
  Precondition:  ALL accumulated fields populated
  Postcondition: final_report with complete research summary
```

In [ ]:
REPORT_SYSTEM = """Compile a final research report from accumulated notes.

Structure:
1. RESEARCH QUESTION
2. METHODOLOGY (how many iterations, sources, claims)
3. KEY FINDINGS (strongest, most-supported claims)
4. CONTRADICTIONS & DEBATES (unresolved disagreements)
5. REMAINING GAPS (acknowledged limitations)
6. CONCLUSION (what the evidence suggests overall)

Write 250-400 words. Be balanced and cite specific sources/claims."""


def compile_report(state: ResearchState) -> dict:
    """Node: Compile final research report from all accumulated data.

    Transition: identify_gaps → compile_report
    Precondition: all fields populated through iterations
    Postcondition: final_report ready
    """
    print("  [NODE] compile_report")

    sources = state.get("sources", [])
    claims = state.get("claims", [])
    notes = state.get("notes", [])

    source_list = "\n".join(
        f"  - {s.get('title', '?')} ({s.get('year', '?')})" for s in sources[:15]
    )
    notes_text = "\n---\n".join(n.get("note", "")[:400] for n in notes)

    report_body = ask(
        f"TOPIC: {state['topic']}\n\n"
        f"ITERATIONS: {state.get('iteration', 0)}\n"
        f"SOURCES ({len(sources)}):\n{source_list}\n\n"
        f"CLAIMS ({len(claims)}):\n"
        f"{chr(10).join(c.get('claim', '')[:100] for c in claims[:15])}\n\n"
        f"ACCUMULATED NOTES:\n{notes_text}\n\n"
        f"REMAINING GAPS:\n{state.get('gaps', 'None identified')}\n\n"
        f"Write the final research report.",
        system=REPORT_SYSTEM,
        temperature=0.3,
    )

    header = (
        f"{'=' * 70}\n"
        f"RESEARCH REPORT\n"
        f"{'=' * 70}\n"
        f"Topic:       {state['topic']}\n"
        f"Iterations:  {state.get('iteration', 0)}\n"
        f"Sources:     {len(sources)}\n"
        f"Claims:      {len(claims)}\n"
        f"Notes:       {len(notes)}\n"
        f"{'=' * 70}\n\n"
    )

    final = header + report_body
    print(f"    Report: {len(final)} chars")
    return {"final_report": final, "current_node": "compile_report"}


print("Node defined: compile_report")
print("  READS:  ALL accumulated state")
print("  WRITES: final_report")

---

# Part C — Routing, Assembly & Checkpointing

## 15. Routing After Gap Analysis

```
  ┌─────────────────────────────────────────────────────────────┐
  │  ROUTING LOGIC                                             │
  ├─────────────────────────────────────────────────────────────┤
  │  IF has_gaps == True AND iteration < MAX_ITERATIONS:       │
  │    → gather_sources (loop back — more research needed)     │
  │    This transition FEEDS gap_queries into the next round   │
  │    The accumulated sources/claims/notes persist via state  │
  │                                                            │
  │  IF has_gaps == False OR iteration >= MAX_ITERATIONS:      │
  │    → compile_report (research is done or capped)           │
  └─────────────────────────────────────────────────────────────┘
```

In [ ]:
MAX_ITERATIONS = 2


def route_after_gaps(state: ResearchState) -> str:
    """Conditional edge: loop for more research or compile report."""
    has_gaps = state.get("has_gaps", False)
    iteration = state.get("iteration", 0)

    if has_gaps and iteration < MAX_ITERATIONS:
        print(f"    [ROUTE] gaps found (iter {iteration}/{MAX_ITERATIONS}) → gather_sources")
        return "gather_sources"

    if iteration >= MAX_ITERATIONS:
        print(f"    [ROUTE] iteration cap ({MAX_ITERATIONS}) → compile_report")
    else:
        print(f"    [ROUTE] no gaps → compile_report")
    return "compile_report"


print(f"Routing: route_after_gaps (max {MAX_ITERATIONS} iterations)")

## 16. Assemble Graph with MemorySaver

**MemorySaver** saves a checkpoint after every node execution. Each checkpoint is keyed by a `thread_id` (config parameter) so you can run multiple research threads independently.

```
  ┌──────────────────────────────────────────────────────────────┐
  │  CHECKPOINTING                                              │
  ├──────────────────────────────────────────────────────────────┤
  │                                                              │
  │  After EACH node:                                            │
  │    MemorySaver.put(thread_id, state_snapshot, node_name)    │
  │                                                              │
  │  This means you can:                                         │
  │    1. Stop a run mid-pipeline                                │
  │    2. Inspect what the state was at any past node            │
  │    3. Resume from the latest checkpoint                      │
  │    4. Fork research: load a checkpoint and explore a branch │
  │                                                              │
  │  MemorySaver stores in Python dict (lost when process ends) │
  │  For disk persistence, use SqliteSaver or PostgresSaver     │
  └──────────────────────────────────────────────────────────────┘
```

In [ ]:
# Create the checkpointer
memory = MemorySaver()

# Build graph
workflow = StateGraph(ResearchState)

workflow.add_node("gather_sources", gather_sources)
workflow.add_node("extract_claims", extract_claims)
workflow.add_node("cross_reference", cross_reference)
workflow.add_node("synthesize_notes", synthesize_notes)
workflow.add_node("identify_gaps", identify_gaps)
workflow.add_node("compile_report", compile_report)

# T1: START → gather_sources
workflow.add_edge(START, "gather_sources")
# T2: gather → extract
workflow.add_edge("gather_sources", "extract_claims")
# T3: extract → cross_reference
workflow.add_edge("extract_claims", "cross_reference")
# T4: cross_reference → synthesize
workflow.add_edge("cross_reference", "synthesize_notes")
# T5: synthesize → identify_gaps
workflow.add_edge("synthesize_notes", "identify_gaps")
# T6: conditional — loop or compile
workflow.add_conditional_edges(
    "identify_gaps",
    route_after_gaps,
    {"gather_sources": "gather_sources", "compile_report": "compile_report"},
)
# T7: compile → END
workflow.add_edge("compile_report", END)

# Compile WITH checkpointer — this enables persistence
app = workflow.compile(checkpointer=memory)

print("Graph compiled with MemorySaver checkpointer!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("Transitions:")
print("  T1: START       → gather_sources    (find sources)")
print("  T2: gather      → extract_claims    (pull claims)")
print("  T3: extract     → cross_reference   (check consistency)")
print("  T4: cross_ref   → synthesize_notes  (write notes)")
print("  T5: synthesize  → identify_gaps     (find what's missing)")
print("  T6: gaps        → gather_sources    (loop if gaps found)")
print("      gaps        → compile_report    (done or at cap)")
print("  T7: compile     → END")

In [ ]:
# Visualize
try:
    print(app.get_graph().draw_mermaid())
except Exception as e:
    print(f"Mermaid not available: {e}")
    print("\nGraph (text):")
    print("  START → gather → extract → cross_ref → synthesize → gaps")
    print("  gaps → gather (loop) | gaps → compile → END")

---

# Part D — Execute with Persistent State

## 17. Run Research Topic 1

The `config` dict contains a `thread_id` — this is the key for checkpoint storage. Different thread IDs = independent research sessions.

In [ ]:
def make_initial_state(research: dict) -> ResearchState:
    return {
        "topic": research["topic"],
        "sub_queries": research["initial_queries"],
        "sources": [],
        "claims": [],
        "notes": [],
        "cross_ref_report": "",
        "synthesis": "",
        "gaps": "",
        "gap_queries": [],
        "has_gaps": False,
        "iteration": 0,
        "final_report": "",
        "current_node": "start",
    }


topic_1 = RESEARCH_TOPICS[0]
config_1 = {"configurable": {"thread_id": "research-rag-hallucinations"}}

print(f"Topic: {topic_1['topic']}")
print(f"Thread ID: {config_1['configurable']['thread_id']}")
print(f"Initial queries: {len(topic_1['initial_queries'])}")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(topic_1),
    config_1,
)

print(f"\nComplete.")
print(f"  Iterations: {result_1['iteration']}")
print(f"  Sources: {len(result_1['sources'])}")
print(f"  Claims: {len(result_1['claims'])}")
print(f"  Notes: {len(result_1['notes'])}")
print(f"  Report: {len(result_1['final_report'])} chars")

## 18. Inspect Accumulated State

Because list fields use `operator.add`, sources and claims from ALL iterations are combined.

In [ ]:
print("ACCUMULATED STATE — Topic 1")
print("=" * 70)

# Sources by iteration
print("\nSOURCES BY ITERATION:")
for s in result_1["sources"]:
    title = s.get("title", "?")[:50]
    year = s.get("year", "?")
    it = s.get("iteration", "?")
    print(f"  [Iter {it}] {title} ({year})")

# Claims by iteration
print(f"\nCLAIMS BY ITERATION ({len(result_1['claims'])} total):")
for c in result_1["claims"][:10]:
    claim = c.get("claim", "?")[:60]
    conf = c.get("confidence", "?")
    it = c.get("iteration", "?")
    print(f"  [Iter {it}] [{conf.upper():>6}] {claim}")

if len(result_1["claims"]) > 10:
    print(f"  ... and {len(result_1['claims']) - 10} more claims")

## 19. View Accumulated Research Notes

Each iteration produced a timestamped note — they're all preserved.

In [ ]:
print("ACCUMULATED RESEARCH NOTES")
print("=" * 70)

for note in result_1["notes"]:
    print(f"\n--- Note (Iteration {note['iteration']}) [{note['timestamp']}] ---")
    print(f"    Sources at this point: {note['sources_count']}")
    print(f"    Claims at this point:  {note['claims_count']}")
    print()
    wrap_print(note["note"][:500])
    if len(note["note"]) > 500:
        print(f"  [...{len(note['note']) - 500} more chars]")

## 20. View Final Report

In [ ]:
print("FINAL RESEARCH REPORT — Topic 1")
print("=" * 70)
wrap_print(result_1["final_report"])

---

# Part E — Checkpoint Inspection & Persistence

## 21. Read Back State from Checkpoints

Because we compiled the graph with `MemorySaver`, every node execution saved a checkpoint. We can read back the state at any point.

In [ ]:
print("CHECKPOINT STATE — Reading back from MemorySaver")
print("=" * 70)

# get_state returns the LATEST checkpoint for this thread
saved_state = app.get_state(config_1)

print(f"Thread ID:    {config_1['configurable']['thread_id']}")
print(f"\nRecovered state:")
print(f"  topic:       {saved_state.values.get('topic', '?')[:50]}")
print(f"  sources:     {len(saved_state.values.get('sources', []))}")
print(f"  claims:      {len(saved_state.values.get('claims', []))}")
print(f"  notes:       {len(saved_state.values.get('notes', []))}")
print(f"  iteration:   {saved_state.values.get('iteration', 0)}")
print(f"  has report:  {bool(saved_state.values.get('final_report', ''))}")
print(f"\nThe full state was recovered from the checkpoint.")
print(f"In production (SqliteSaver), this would survive process restarts.")

## 22. Inspect Checkpoint History

Every node execution created a snapshot. We can walk through the research timeline.

In [ ]:
print("CHECKPOINT HISTORY — State Evolution")
print("=" * 80)

history = list(app.get_state_history(config_1))
print(f"Total checkpoints: {len(history)}\n")

print(f"{'#':>3} {'Node':22} {'Sources':>8} {'Claims':>8} {'Notes':>6} {'Iter':>5}")
print("-" * 60)

for i, snapshot in enumerate(reversed(history)):
    vals = snapshot.values
    node = vals.get("current_node", "init")
    n_src = len(vals.get("sources", []))
    n_cl = len(vals.get("claims", []))
    n_nt = len(vals.get("notes", []))
    it = vals.get("iteration", 0)
    print(f"  {i:>2}  {node:<22} {n_src:>8} {n_cl:>8} {n_nt:>6} {it:>5}")

print("\nNotice how sources, claims, notes GROW as the pipeline progresses.")
print("Each checkpoint is a full snapshot — the graph can resume from any one.")

## 23. Streaming — Watch Accumulation in Real Time

In [ ]:
print("STREAMING EXECUTION — Topic 1 (re-run)")
print("=" * 75)

stream_config = {"configurable": {"thread_id": "stream-demo"}}
step = 0

for chunk in app.stream(make_initial_state(topic_1), stream_config):
    step += 1
    for node_name, node_output in chunk.items():
        details = []
        for k, v in node_output.items():
            if k == "current_node":
                continue
            if isinstance(v, list):
                details.append(f"{k}: +{len(v)} items")
            elif isinstance(v, str) and len(v) > 50:
                details.append(f"{k}: {len(v)} chars")
            elif isinstance(v, bool):
                details.append(f"{k}: {v}")
            elif isinstance(v, int):
                details.append(f"{k}: {v}")
            else:
                details.append(f"{k}: {v}")
        print(f"  Step {step}: {node_name:<22} → {', '.join(details)}")

print(f"\nTotal steps: {step}")

---

# Part F — Second Topic & Comparison

## 24. Run Research Topic 2

Different `thread_id` — completely independent research session.

In [ ]:
topic_2 = RESEARCH_TOPICS[1]
config_2 = {"configurable": {"thread_id": "research-transfer-learning"}}

print(f"Topic: {topic_2['topic']}")
print(f"Thread ID: {config_2['configurable']['thread_id']}")
print("=" * 70)

result_2 = app.invoke(
    make_initial_state(topic_2),
    config_2,
)

print(f"\nComplete.")
print(f"  Iterations: {result_2['iteration']}")
print(f"  Sources: {len(result_2['sources'])}")
print(f"  Claims: {len(result_2['claims'])}")
print(f"  Notes: {len(result_2['notes'])}")
print(f"  Report: {len(result_2['final_report'])} chars")

## 25. Cross-Thread Comparison

In [ ]:
print("RESEARCH COMPARISON")
print("=" * 85)

df = pd.DataFrame([
    {
        "Topic": RESEARCH_TOPICS[i]["topic"][:40],
        "Thread ID": [config_1, config_2][i]["configurable"]["thread_id"],
        "Iterations": r["iteration"],
        "Sources": len(r["sources"]),
        "Claims": len(r["claims"]),
        "Notes": len(r["notes"]),
        "Report (chars)": len(r["final_report"]),
    }
    for i, r in enumerate([result_1, result_2])
])

print(df.to_string(index=False))

# Verify thread isolation
print("\nTHREAD ISOLATION CHECK:")
state_t1 = app.get_state(config_1).values
state_t2 = app.get_state(config_2).values
print(f"  Thread 1 topic: {state_t1.get('topic', '?')[:40]}")
print(f"  Thread 2 topic: {state_t2.get('topic', '?')[:40]}")
print(f"  Are they different? {state_t1.get('topic') != state_t2.get('topic')}")
print("  → Each thread_id maintains independent state.")

## 26. View Topic 2 Report

In [ ]:
print("FINAL RESEARCH REPORT — Topic 2")
print("=" * 70)
wrap_print(result_2["final_report"])

---

# Part G — Accumulation & Memory Analysis

## 27. Accumulation Growth Chart

In [ ]:
print("ACCUMULATION GROWTH — Topic 1")
print("=" * 70)

# Walk through checkpoints to see how lists grew
history = list(app.get_state_history(config_1))

print(f"{'Step':>5} {'Node':22} {'Sources':>10} {'Claims':>10} {'Notes':>8}")
print("-" * 60)

seen_nodes = set()
for i, snap in enumerate(reversed(history)):
    vals = snap.values
    node = vals.get("current_node", "init")
    key = f"{i}-{node}"
    if key not in seen_nodes:
        seen_nodes.add(key)
        n_src = len(vals.get("sources", []))
        n_cl = len(vals.get("claims", []))
        n_nt = len(vals.get("notes", []))
        src_bar = "█" * min(n_src, 20)
        cl_bar = "█" * min(n_cl, 20)
        print(f"  {i:>3}  {node:<22} {n_src:>5} {src_bar:<10} {n_cl:>5} {cl_bar:<10} {n_nt:>4}")

print("\nNotice: sources and claims only GROW — they never shrink.")
print("This is the operator.add reducer at work.")

## 28. Memory Types Compared

```
  ╔════════════════════════════════════════════════════════════════════╗
  ║  MEMORY TYPE COMPARISON (as seen in this notebook)               ║
  ╠═══════════════════╦══════════════════════╦═══════════════════════╣
  ║  Concept           ║  Implementation      ║  Lifetime             ║
  ╠═══════════════════╬══════════════════════╬═══════════════════════╣
  ║  Graph State       ║  ResearchState dict  ║  Single invoke() call ║
  ║  Accumulation      ║  operator.add lists  ║  Grows within state   ║
  ║  Checkpoint        ║  MemorySaver         ║  Across invoke() calls║
  ║  Thread Isolation   ║  thread_id config    ║  Per research topic   ║
  ╠═══════════════════╬══════════════════════╬═══════════════════════╣
  ║  NOT shown here:   ║                      ║                       ║
  ║  SqliteSaver       ║  Writes to .db file  ║  Survives restarts    ║
  ║  PostgresSaver     ║  Postgres database   ║  Production-grade     ║
  ╚═══════════════════╩══════════════════════╩═══════════════════════╝

  WHY USE operator.add INSTEAD OF OVERWRITING LISTS?
  ─────────────────────────────────────────────────
  Without it: Node returns {sources: [new_src]}
              → state[sources] = [new_src]  (old sources LOST!)

  With it:    Node returns {sources: [new_src]}
              → state[sources] = old_sources + [new_src]  (accumulated!)

  This is essential for iterative research where each loop adds data.
```

## 29. Exercises

### Exercise 1: Resume from Checkpoint
After the first run completes, add a new query to an existing thread:
```python
app.invoke({"sub_queries": ["new query here"], "gap_queries": ["new query"], "has_gaps": True, "iteration": 0}, config_1)
```
Check if the accumulated sources/claims from Run 1 are still present.

### Exercise 2: Source Deduplication
Add logic to `gather_sources` that checks for duplicate titles in the existing `state["sources"]` and skips sources with matching titles. This prevents the same paper from being "found" multiple times across iterations.

### Exercise 3: Claim Confidence Aggregation
Add a `confidence_summary` node after `cross_reference` that counts claims by confidence level (high/medium/low) and by iteration. Use this to decide whether the research needs more high-confidence evidence.

### Mini Challenge: SqliteSaver
Replace `MemorySaver` with `SqliteSaver` (from `langgraph.checkpoint.sqlite`). Run the pipeline, close the notebook, reopen it, and verify you can read back the full state from the SQLite file.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Topics researched:   {len(RESEARCH_TOPICS)}")
print(f"Topic 1 — Sources: {len(result_1['sources'])}, Claims: {len(result_1['claims'])}, Notes: {len(result_1['notes'])}")
print(f"Topic 2 — Sources: {len(result_2['sources'])}, Claims: {len(result_2['claims'])}, Notes: {len(result_2['notes'])}")
print()
print("State transitions:")
print("  T1: START      → gather_sources    + sources (APPEND)")
print("  T2: gather     → extract_claims    + claims (APPEND)")
print("  T3: extract    → cross_reference   + cross_ref_report")
print("  T4: cross_ref  → synthesize_notes  + notes (APPEND), synthesis")
print("  T5: synthesize → identify_gaps     + gaps, has_gaps, iteration")
print("  T6: gaps       → gather_sources    (loop if gaps found)")
print("      gaps       → compile_report    (done or at cap)")
print("  T7: compile    → END               + final_report")
print()
print("Key concepts demonstrated:")
print("  • operator.add reducers — lists ACCUMULATE across iterations")
print("  • MemorySaver checkpointer — state persists across runs")
print("  • thread_id isolation — independent research sessions")
print("  • Checkpoint history — inspect state at any past node")
print("  • Research loop — gaps feed new queries into gather_sources")
print("  • Iteration tracking — each source/claim tagged with its round")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Graph state ≠ persistence** — state flows during a run; checkpoints save it across runs |
| 2 | **operator.add makes lists accumulate** — each node APPENDS, never overwrites |
| 3 | **MemorySaver saves after every node** — you can resume from any checkpoint |
| 4 | **thread_id isolates sessions** — multiple research threads don't mix |
| 5 | **Iteration tagging is essential** — mark each source/claim with its loop round |
| 6 | **Cross-reference uses ALL data** — not just the latest batch, but everything accumulated |
| 7 | **Gap-driven loops make research iterative** — like a real researcher going deeper |
| 8 | **Cap all loops** — MAX_ITERATIONS prevents unbounded research |
| 9 | **Checkpoint history is an audit trail** — see exactly how state evolved |
| 10 | **For production, use SqliteSaver or PostgresSaver** — MemorySaver is process-scoped |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*